# 🏆 Challenge : Predict Conversions
**Projet Jedha — Classification supervisée · Compétition ML style Kaggle**

Structure basée sur le template officiel Jedha (`02-Conversion_rate_challenge_template.ipynb`)

**Métrique officielle : F1-score**

---
## Plan
- **Partie 1** : EDA + Prétraitement + Modèle baseline
- **Partie 2** : Amélioration du F1 (Random Forest + GridSearch)
- **Partie 3** : Prédictions sur fichier test → soumission CSV
- **Partie 4** : Analyse du meilleur modèle → recommandations

# Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
print('✅ Imports OK')

# Read file with labels

In [ ]:
data = pd.read_csv('conversion_data_train.csv')
print('Set with labels (our train+test) :', data.shape)
data.head()

# Explore dataset

> ⚠️ Le dataset est grand (284 580 lignes). On travaille sur un **échantillon de 10 000 lignes** pour les visualisations, comme recommandé par Jedha.

In [ ]:
# Jedha recommande de créer un échantillon avant les visualisations sur ce grand dataset
data_sample = data.sample(10000, random_state=42)
print(f'Taille de l\'échantillon de visualisation : {data_sample.shape}')

In [ ]:
# Types et valeurs manquantes
print('=== Info dataset ===')
data.info()
print()
print('=== Valeurs manquantes ===')
print(data.isnull().sum())

In [ ]:
data.describe()

## ⚠️ Déséquilibre de classes — problème central

Seulement ~3.2% des visiteurs convertissent. Un modèle qui prédit toujours 0 aurait 96.8% d'accuracy — inutile.  
C'est pourquoi la métrique est le **F1-score**, qui tient compte des faux positifs ET des faux négatifs.

In [ ]:
conv_counts = data['converted'].value_counts()
conv_pct = data['converted'].value_counts(normalize=True) * 100
print('Distribution de la variable cible :')
print(pd.DataFrame({'Nombre': conv_counts, 'Pourcentage (%)': conv_pct.round(2)}))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Non converti (0)', 'Converti (1)'], conv_counts, color=['#e74c3c', '#06D6A0'], edgecolor='white')
for bar, pct in zip(bars, conv_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f'{pct:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_title('Déséquilibre de classes — variable cible converted')
ax.set_ylabel('Nombre de lignes')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisations sur l'échantillon (comme recommandé par Jedha pour ce grand dataset)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in [(0, '#e74c3c'), (1, '#06D6A0')]:
    subset = data_sample[data_sample['converted'] == label]['total_pages_visited']
    axes[0].hist(subset, bins=30, alpha=0.6, label=f'converted={label}', color=color)
axes[0].set_title('Pages visitées selon conversion (échantillon 10K)')
axes[0].set_xlabel('total_pages_visited')
axes[0].legend()

# Sur tout le dataset : taux de conversion par pages
conv_by_pages = data.groupby('total_pages_visited')['converted'].mean() * 100
axes[1].bar(conv_by_pages.index, conv_by_pages.values, color='#4CC9F0', edgecolor='white')
axes[1].set_title('Taux de conversion par nombre de pages visitées')
axes[1].set_xlabel('total_pages_visited')
axes[1].set_ylabel('Taux de conversion (%)')

plt.tight_layout()
plt.show()

print('Pages visitées moyennes — non convertis :', data[data['converted']==0]['total_pages_visited'].mean().round(2))
print('Pages visitées moyennes — convertis     :', data[data['converted']==1]['total_pages_visited'].mean().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Taux de conversion par pays
conv_country = data.groupby('country')['converted'].mean().sort_values(ascending=False) * 100
axes[0].bar(conv_country.index, conv_country.values, color='#4CC9F0', edgecolor='white')
axes[0].set_title('Taux de conversion par pays')
axes[0].set_ylabel('Taux (%)')

# Taux de conversion par source
conv_source = data.groupby('source')['converted'].mean().sort_values(ascending=False) * 100
axes[1].bar(conv_source.index, conv_source.values, color='#FFD166', edgecolor='white')
axes[1].set_title('Taux de conversion par source')
axes[1].set_ylabel('Taux (%)')

# Taux de conversion new_user vs returning
conv_new = data.groupby('new_user')['converted'].mean() * 100
axes[2].bar(['Ancien utilisateur (0)', 'Nouvel utilisateur (1)'], conv_new.values, color='#EF476F', edgecolor='white')
axes[2].set_title('Taux de conversion : new_user')
axes[2].set_ylabel('Taux (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution de l'âge — sur l'échantillon
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in [(0, '#e74c3c'), (1, '#06D6A0')]:
    subset = data_sample[data_sample['converted'] == label]['age']
    axes[0].hist(subset, bins=40, alpha=0.6, label=f'converted={label}', color=color)
axes[0].set_title('Distribution âge selon conversion (échantillon 10K)')
axes[0].set_xlabel('age')
axes[0].legend()

axes[1].boxplot([data[data['converted']==0]['age'], data[data['converted']==1]['age']],
                labels=['Non converti', 'Converti'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot âge — outlier visible (age=123)')
axes[1].set_ylabel('age')

plt.tight_layout()
plt.show()
print(f"Lignes avec age > 100 : {(data['age'] > 100).sum()}")

## 🧹 Nettoyage — suppression des outliers sur l'âge

> Un âge de 123 ans est biologiquement impossible. On applique la règle ±3σ.

In [ ]:
print(f'Avant nettoyage : {data.shape[0]} lignes')

mean_age = data['age'].mean()
std_age  = data['age'].std()
lower = mean_age - 3 * std_age
upper = mean_age + 3 * std_age
print(f'Intervalle âge valide : [{lower:.1f}, {upper:.1f}]')

data = data[data['age'].between(lower, upper)].reset_index(drop=True)
print(f'Après nettoyage : {data.shape[0]} lignes')

# Make your model

## Choose variables and create train/test sets

**Jedha template** commence avec `total_pages_visited` seul. On étend à toutes les variables disponibles.

> **`stratify=y`** est obligatoire ici : avec 3.23% de positifs, un split aléatoire déséquilibrerait les ensembles.

In [ ]:
# Comme dans le template Jedha — on déclare explicitement les features
features_list     = ['country', 'age', 'new_user', 'source', 'total_pages_visited']
cat_features      = ['country', 'source']
num_features      = ['age', 'new_user', 'total_pages_visited']
target_variable   = 'converted'

X = data.loc[:, features_list]
Y = data.loc[:, target_variable]

print('Explanatory variables :', X.columns.tolist())
print('Target variable :', target_variable)

In [ ]:
# Divide dataset into Train set & Test set
print('Dividing into train and test sets...')
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    stratify=Y  # OBLIGATOIRE avec déséquilibre de classes
)
print('...Done.')
print(f'X_train : {X_train.shape} | X_test : {X_test.shape}')
print(f'Taux positifs train : {Y_train.mean()*100:.2f}%')
print(f'Taux positifs test  : {Y_test.mean()*100:.2f}%')

## Training pipeline

> **Règle du template Jedha** : le prétraitement est appris (`.fit_transform()`) **uniquement sur le train set**.
> Sur le test set on appelle uniquement `.transform()` (pas `.fit_transform()`).
> Le Pipeline scikit-learn garantit cette règle automatiquement.

In [ ]:
print('Encoding categorical features and standardizing numerical features...')

# Pipeline numériques : imputation médiane + StandardScaler
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Pipeline catégorielles : imputation mode + OneHotEncoder
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer : applique chaque pipeline aux bonnes colonnes
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

print('...Done')
print('Preprocessor défini : StandardScaler + OneHotEncoder dans un Pipeline')

In [ ]:
# === BASELINE : Régression Logistique (comme dans le template Jedha) ===
print('Train baseline model (Logistic Regression)...')

# class_weight='balanced' : compense le déséquilibre — INDISPENSABLE
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

lr_pipeline.fit(X_train, Y_train)
print('...Done.')

# Predictions on training set
print('\nPredictions on training set...')
Y_train_pred_lr = lr_pipeline.predict(X_train)
print('...Done.')

## Test pipeline

> On utilise le **même préprocesseur**, mais `.predict()` appelle `.transform()` en interne (pas `.fit_transform()`).  
> Le Pipeline garantit qu'aucune information du test set ne "pollue" le prétraitement.

In [ ]:
# Predictions on test set
print('Predictions on test set...')
Y_test_pred_lr = lr_pipeline.predict(X_test)
print('...Done.')

## Evaluate performances

> ⚠️ Jedha impose : **utiliser la même métrique que le leaderboard** = F1-score

In [ ]:
# WARNING : Use the same score as Kaggle/Jedha leaderboard = f1_score
print('=== Baseline — Régression Logistique ===')
print(f'f1-score on train set : {f1_score(Y_train, Y_train_pred_lr):.4f}')
print(f'f1-score on test set  : {f1_score(Y_test,  Y_test_pred_lr):.4f}')
print()
print('Confusion matrix on train set :')
print(confusion_matrix(Y_train, Y_train_pred_lr))
print()
print('Confusion matrix on test set :')
print(confusion_matrix(Y_test, Y_test_pred_lr))
print()
print(classification_report(Y_test, Y_test_pred_lr, target_names=['Non converti', 'Converti']))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    Y_test, Y_test_pred_lr,
    display_labels=['Non converti (0)', 'Converti (1)'],
    cmap='Blues', ax=ax
)
ax.set_title('Matrice de confusion — Logistic Regression baseline')
plt.tight_layout()
plt.show()

---
# PARTIE 2 — Améliorer le F1-score

Le modèle baseline atteint ~0.508. Essayons de battre ce score avec un **Random Forest** et **GridSearchCV**.

> Le Random Forest est un modèle non-linéaire (ensemble d'arbres de décision) mieux adapté pour capturer des interactions complexes entre variables. `class_weight='balanced'` fonctionne de la même façon qu'en logistic regression.

In [ ]:
# === MODÈLE AMÉLIORÉ : Random Forest ===
print('Train Random Forest...')

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, Y_train)
print('...Done.')

Y_train_pred_rf = rf_pipeline.predict(X_train)
Y_test_pred_rf  = rf_pipeline.predict(X_test)

print(f'\nf1-score on train set : {f1_score(Y_train, Y_train_pred_rf):.4f}')
print(f'f1-score on test set  : {f1_score(Y_test,  Y_test_pred_rf):.4f}')
print()
print('Confusion matrix on test set :')
print(confusion_matrix(Y_test, Y_test_pred_rf))

In [ ]:
# === OPTIMISATION : GridSearchCV sur Random Forest ===
# On optimise directement le F1-score (scoring='f1') — pas l'accuracy !
print('GridSearchCV (patience, ça prend quelques minutes)...')

rf_gs_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

param_grid = {
    'model__n_estimators':     [100],
    'model__max_depth':        [10, 20],
    'model__min_samples_leaf': [5, 10]
}

grid_search = GridSearchCV(
    rf_gs_pipeline,
    param_grid,
    cv=3,
    scoring='f1',   # ← même métrique que le leaderboard !
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, Y_train)

print(f'\n✅ Meilleurs paramètres : {grid_search.best_params_}')
print(f'   F1 CV train          : {grid_search.best_score_:.4f}')

best_model = grid_search.best_estimator_
Y_train_pred_best = best_model.predict(X_train)
Y_test_pred_best  = best_model.predict(X_test)

print(f'   F1 test set          : {f1_score(Y_test, Y_test_pred_best):.4f}')
print()
print(classification_report(Y_test, Y_test_pred_best, target_names=['Non converti', 'Converti']))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    Y_test, Y_test_pred_best,
    display_labels=['Non converti (0)', 'Converti (1)'],
    cmap='Greens', ax=ax
)
ax.set_title('Matrice de confusion — Random Forest (meilleur modèle)')
plt.tight_layout()
plt.show()

In [ ]:
# Comparatif des modèles
print('=== Comparatif F1-score ===')
results = {
    'Modèle': ['LogisticRegression (balanced)', 'RandomForest (balanced)', 'RandomForest (GridSearch)'],
    'F1 Train': [
        f1_score(Y_train, Y_train_pred_lr),
        f1_score(Y_train, Y_train_pred_rf),
        f1_score(Y_train, Y_train_pred_best)
    ],
    'F1 Test': [
        f1_score(Y_test, Y_test_pred_lr),
        f1_score(Y_test, Y_test_pred_rf),
        f1_score(Y_test, Y_test_pred_best)
    ]
}
results_df = pd.DataFrame(results)
results_df[['F1 Train', 'F1 Test']] = results_df[['F1 Train', 'F1 Test']].round(4)
print(results_df.to_string(index=False))

---
# PARTIE 3 — Prédictions sur le fichier test → soumission

## Entraîner le meilleur modèle sur TOUTES les données

> **Instruction Jedha (template)** : avant de prédire sur le fichier de soumission, on ré-entraîne le meilleur modèle sur l'ensemble complet (train + test) pour utiliser le maximum de données.

In [ ]:
# Concatenate our train and test sets to train on all labeled data
# (comme recommandé dans le template Jedha)
X_all = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
Y_all = pd.concat([Y_train, Y_test], axis=0).reset_index(drop=True)

print(f'Données totales pour entraînement final : {X_all.shape[0]} lignes')

# Reprendre les meilleurs hyperparamètres trouvés par GridSearch
best_params = grid_search.best_params_
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=best_params['model__n_estimators'],
        max_depth=best_params['model__max_depth'],
        min_samples_leaf=best_params['model__min_samples_leaf'],
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

print('Entraînement sur toutes les données...')
final_pipeline.fit(X_all, Y_all)
print('...Done.')

In [ ]:
# Read data without labels
data_without_labels = pd.read_csv('conversion_data_test.csv')
print('Prediction set (without labels) :', data_without_labels.shape)

# WARNING : features_list must be the same as in training
X_without_labels = data_without_labels.loc[:, features_list]
print('Features utilisées :', features_list)

# Make predictions
print('\nPredictions on test set (without labels)...')
Y_predictions = final_pipeline.predict(X_without_labels)
print('...Done.')
print(f'Conversions prédites : {Y_predictions.sum()} ({Y_predictions.mean()*100:.2f}%)')

In [ ]:
# Make predictions and dump to file
# WARNING : FILE NAME FORMAT = 'conversion_data_test_predictions_[NOM-MODELE].csv'
# WARNING : CSV WITH ONE COLUMN NAMED 'converted' AND NO INDEX !

output = pd.DataFrame({'converted': Y_predictions})
output.to_csv('conversion_data_test_predictions_RF-GridSearch.csv', index=False)

print('✅ Fichier créé : conversion_data_test_predictions_RF-GridSearch.csv')
print(f'   Nombre de prédictions : {len(Y_predictions)}')
output.head(10)

---
# PARTIE 4 — Analyse du meilleur modèle & Recommandations

## Analyse des feature importances

> Le Random Forest calcule l'**importance Gini** de chaque feature : combien elle réduit l'impureté dans les arbres. Une feature à haute importance est utilisée souvent et tôt dans les arbres.

In [ ]:
# Récupérer les noms de features après transformation
preprocessor_fitted = final_pipeline.named_steps['preprocessor']
cat_encoder = preprocessor_fitted.transformers_[1][1].named_steps['encoder']
cat_names = cat_encoder.get_feature_names_out(cat_features).tolist()
all_feature_names = num_features + cat_names

importances = final_pipeline.named_steps['model'].feature_importances_
feat_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 6))
colors = ['#FFD166' if i == 0 else '#4CC9F0' for i in range(len(feat_df))]
plt.barh(feat_df['Feature'][::-1], feat_df['Importance'][::-1], color=colors[::-1], edgecolor='white')
plt.xlabel('Importance (Gini)')
plt.title('Feature importances — Random Forest (modèle final)')
plt.tight_layout()
plt.show()

print('Top 5 features :')
print(feat_df.head(5).to_string(index=False))

In [ ]:
# Taux de conversion par tranche de pages visitées
data['pages_bin'] = pd.cut(data['total_pages_visited'],
                            bins=[0, 3, 7, 15, 30],
                            labels=['1-3', '4-7', '8-15', '16+'])
conv_by_bin = data.groupby('pages_bin', observed=True)['converted'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

conv_by_bin.plot(kind='bar', ax=axes[0], color='#4CC9F0', edgecolor='white', rot=0)
axes[0].set_title('Taux de conversion par volume de pages visitées')
axes[0].set_xlabel('Pages visitées')
axes[0].set_ylabel('Taux de conversion (%)')
for i, v in enumerate(conv_by_bin):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')

conv_country_source = data.groupby(['country', 'source'])['converted'].mean() * 100
conv_country_source.unstack().plot(kind='bar', ax=axes[1], edgecolor='white', rot=30)
axes[1].set_title('Taux de conversion : pays × source')
axes[1].set_ylabel('Taux de conversion (%)')
axes[1].legend(title='Source')

plt.tight_layout()
plt.show()

print('\nTaux de conversion par pays :')
print((data.groupby('country')['converted'].mean() * 100).round(2).sort_values(ascending=False))
print('\nTaux de conversion par new_user :')
print((data.groupby('new_user')['converted'].mean() * 100).round(2))

## 💡 Recommandations métier

### Ce que le modèle révèle

**1. `total_pages_visited` = 76% de l'importance Gini**  
C'est de loin le signal dominant. Les visiteurs qui consultent beaucoup de pages sont bien plus susceptibles de convertir.  
**→ Recommandation : améliorer la navigation, proposer des articles similaires, augmenter la qualité du contenu pour retenir les visiteurs plus longtemps.**

**2. Anciens utilisateurs (`new_user=0`) convertissent 5× plus (7.2% vs 1.4%)**  
La rétention est plus rentable que l'acquisition de nouveaux visiteurs.  
**→ Recommandation : email de rappel, retargeting, push notifications pour les anciens visiteurs.**

**3. Forte disparité géographique : Allemagne 6.3%, UK 5.3%, Chine 0.13%**  
**→ Recommandation : concentrer les budgets marketing sur les marchés Allemagne et UK.**

**4. La source de trafic a peu d'impact (< 1% d'importance)**  
**→ Conclusion : optimiser la source d'acquisition est secondaire. L'engagement sur site est le levier principal.**